# Pipeline y verificaciones de PuntoBot

Comprobaciones previas al entrenamiento final, adaptadas de la guía de pipeline a un proyecto de NLP.

En Colab, autoriza un secreto llamado `GITHUB_TOKEN` con lectura de este repositorio privado. En tu PC, instala `requirements-keras.txt` y abre el notebook desde el repositorio. Nunca pegues el token en una celda. Los datos de demostración son sintéticos; `test.csv` se mantiene sellado.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, io, urllib.request, subprocess

# En Colab: abre el notebook desde GitHub y autoriza el secreto GITHUB_TOKEN.
# El token se usa solo en una cabecera HTTP y nunca se imprime ni se guarda.
if Path('/content').exists() and not Path('config.yaml').exists():
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    request = urllib.request.Request(
        'https://api.github.com/repos/rebirth18-09/puntobot-ia/zipball/main',
        headers={'Authorization': 'Bearer ' + token, 'Accept': 'application/vnd.github+json'})
    with urllib.request.urlopen(request) as response:
        contenido = response.read()
    del token, request
    destino = Path('/content/puntobot-ia')
    destino.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(contenido)) as archivo:
        for item in archivo.infolist():
            target = (destino / item.filename).resolve()
            if not target.is_relative_to(destino.resolve()):
                raise ValueError('Ruta inválida en el ZIP')
        archivo.extractall(destino)
    os.chdir(next(p for p in destino.iterdir() if (p / 'config.yaml').exists()))
RAIZ = Path.cwd()
if not (RAIZ / 'config.yaml').exists() and (RAIZ.parent / 'config.yaml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / 'config.yaml').exists(), 'Ejecuta el notebook desde la carpeta del repositorio'
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
if Path('/content').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-keras.txt'], check=True)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import tensorflow as tf
import keras
import numpy as np
import yaml
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass
config = yaml.safe_load((RAIZ / 'config.yaml').read_text('utf-8'))
keras.utils.set_random_seed(config['entrenamiento']['semilla'])
print('TensorFlow', tf.__version__, 'Keras', keras.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))

## Verificar

Preserva las particiones, busca IDs y textos repetidos entre conjuntos, aprende TF-IDF y pesos solo de train, comprueba forma y rango, probabilidades, misma preparación en inferencia, contrato JSON, entradas inválidas y serialización. Sobreajusta una red descartable de Keras sobre un lote de 32 consultas; el modelo elegido del proyecto sigue siendo regresión logística.

In [ ]:
from scripts.verificar import verificar
resultados = verificar(incluir_keras=True, guardar=False)
print(json.dumps(resultados, ensure_ascii=False, indent=2))

## Ver nueve consultas con sus clases

In [ ]:
from src.data.division import cargar_particiones
train, val, auditoria = cargar_particiones(config, RAIZ)
print(train[['texto_consulta','intencion']].head(9).to_string(index=False))

## Interpretación

La diferencia entre pipeline e inferencia debe ser 0. La salida tiene cuatro probabilidades que suman 1. La pérdida inicial de la red es del orden de ln(4); el lote debe poder memorizarse. El baseline siempre predice la clase mayoritaria de train. Se comparan accuracy y F1 macro en val.

Las frases sintéticas comparten plantillas, incluso sin duplicados exactos. Esa dependencia puede inflar las métricas; la etapa 7 debe usar consultas reales y una división por origen, sesión o plantilla si se dispone de esos identificadores. No se inventa un ID de paciente para datos de texto que no lo tienen.